# Real-Time Face Recognition System

### Pipeline:
Face Detection (HOG) → Face Encoding (dlib) → Face Matching → Real-Time Recognition

---

## Project Overview

This project performs real-time face recognition using a webcam.

The system:
- Detects faces in video frames
- Extracts facial encodings
- Compares detected faces with stored encodings
- Displays the recognized person's name in real time

The project uses:
- OpenCV for video processing
- face_recognition / dlib for face detection and recognition
- Pickle for local database storage

---

## Notebook Workflow

1. **Cell 1** → Install required libraries  
2. **Cell 2** → Create project structure and database  
3. **Cell 3** → Register multiple people  
4. **Cell 4** → Run real-time face recognition  
5. **Cell 5** → Delete a person from the database  
6. **Cell 6** → Display database contents

## Cell 1 - Install Required Libraries

In [ ]:
# Run this cell only once
import subprocess

packages = [
    'opencv-python',
    'cmake',
    'dlib',
    'face-recognition',
    'ultralytics',
    'numpy'
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.run(['pip', 'install', pkg, '-q'], check=True)

print('All libraries installed successfully.')

## Cell 2 - Create Project Structure

In [ ]:
import os
import pickle

# Create database folder if it does not exist
os.makedirs('database', exist_ok=True)

# Create empty encodings file if it does not exist
if not os.path.exists('encodings.pkl'):
    with open('encodings.pkl', 'wb') as f:
        pickle.dump({'names': [], 'encodings': []}, f)
    print('Created new encodings.pkl')
else:
    with open('encodings.pkl', 'rb') as f:
        data = pickle.load(f)
    print(f'encodings.pkl already exists - {len(set(data["names"]))} person(s) registered')

print('\nProject structure:')
print('project/')
print('|-- database/      <- add a folder per person containing their photos')
print('|-- encodings.pkl  <- face encodings database')
print('|-- notebook.ipynb')

## Cell 3 - Add All Team Members at Once

Make sure each person has a folder inside `database/` with their photos before running.

Before running:
- Create a folder inside `database/` named after the person
- Add 2 to 5 photos of that person inside the folder
- Example: `database/Ahmed/photo1.jpg`

In [ ]:
import face_recognition
import pickle
import os

def load_all_team(database_path='database/'):
    # Overwrites existing encodings with a fresh scan of all folders
    data = {'names': [], 'encodings': []}

    for person_name in os.listdir(database_path):
        person_folder = os.path.join(database_path, person_name)

        if not os.path.isdir(person_folder):
            continue

        print(f'Processing: {person_name}')

        for img_file in os.listdir(person_folder):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            img_path  = os.path.join(person_folder, img_file)
            image     = face_recognition.load_image_file(img_path)
            encodings = face_recognition.face_encodings(image)

            if encodings:
                data['encodings'].append(encodings[0])
                data['names'].append(person_name)
                print(f'  {img_file} - OK')
            else:
                print(f'  {img_file} - No face detected, skipped.')

    with open('encodings.pkl', 'wb') as f:
        pickle.dump(data, f)

    print(f'\nDone. {len(set(data["names"]))} person(s), {len(data["encodings"])} encoding(s) saved.')

load_all_team()

## Cell 4 - Real-Time Face Recognition

Press `q` to stop the camera.

In [ ]:
import cv2
import face_recognition
import numpy as np
import pickle

# Load Database
with open('encodings.pkl', 'rb') as f:
    data = pickle.load(f)

known_encodings = data['encodings']
known_names     = data['names']

print(f'Loaded {len(set(known_names))} people')
print('Camera is running - press q to exit')

# Open Camera 
cap = cv2.VideoCapture(0)

# Reduce camera resolution for better performance
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

frame_count = 0
face_names  = []
face_boxes  = []

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    # Process every 4th frame for smoother performance
    if frame_count % 4 == 0:

        # Resize frame to 1/4 size
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)

        # Convert BGR to RGB
        rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

        # Faster face detection using HOG
        locations = face_recognition.face_locations(
            rgb_small,
            model='hog'
        )

        encodings = face_recognition.face_encodings(
            rgb_small,
            locations
        )

        face_names = []
        face_boxes = []

        for (top, right, bottom, left), encoding in zip(locations, encodings):

            name = 'Unknown'

            distances = face_recognition.face_distance(
                known_encodings,
                encoding
            )

            if len(distances) > 0:

                best_match = np.argmin(distances)

                if distances[best_match] < 0.5:
                    name = known_names[best_match]

            # Scale locations back to original frame size
            top    *= 4
            right  *= 4
            bottom *= 4
            left   *= 4

            face_boxes.append((left, top, right, bottom))
            face_names.append(name)

    # Draw Results
    for (left, top, right, bottom), name in zip(face_boxes, face_names):

        color = (0, 200, 0) if name != 'Unknown' else (0, 0, 220)

        cv2.rectangle(
            frame,
            (left, top),
            (right, bottom),
            color,
            2
        )

        cv2.rectangle(
            frame,
            (left, bottom - 35),
            (right, bottom),
            color,
            cv2.FILLED
        )

        cv2.putText(
            frame,
            name,
            (left + 6, bottom - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 255),
            2
        )

    cv2.imshow('Face Recognition', frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup 
cap.release()
cv2.destroyAllWindows()

print('Camera closed')

## Cell 5 - Delete a Person from the Database

In [ ]:
import pickle

# Change this to the name of the person you want to remove
PERSON_TO_DELETE = ""      # ====> Write the name of folder to Delete from Database

with open('encodings.pkl', 'rb') as f:
    data = pickle.load(f)

before   = len(data['names'])
filtered = [(n, e) for n, e in zip(data['names'], data['encodings']) if n != PERSON_TO_DELETE]

if filtered:
    names, encodings  = zip(*filtered)
    data['names']     = list(names)
    data['encodings'] = list(encodings)
else:
    data = {'names': [], 'encodings': []}

with open('encodings.pkl', 'wb') as f:
    pickle.dump(data, f)

after = len(data['names'])
print(f'Removed {PERSON_TO_DELETE} - {before - after} encoding(s) deleted.')
print(f'Database now has {len(set(data["names"]))} person(s).')

Removed Ahmed - 0 encoding(s) deleted.
Database now has 1 person(s).


## Cell 6 - View Database Contents

In [8]:
import pickle
from collections import Counter

with open('encodings.pkl', 'rb') as f:
    data = pickle.load(f)

if not data['names']:
    print('Database is empty.')
else:
    counts = Counter(data['names'])
    print('Database contents:')
    print('-' * 35)
    for name, count in counts.items():
        print(f'  {name}: {count} encoding(s)')
    print('-' * 35)
    print(f'  Total: {len(set(data["names"]))} person(s), {len(data["encodings"])} encoding(s)')

Database contents:
-----------------------------------
  Omar: 6 encoding(s)
-----------------------------------
  Total: 1 person(s), 6 encoding(s)
